In [1]:
# Install packages

!pip install pandas numpy

# HTTP requests for API ingestion
!pip install requests

# Embedding model
!pip install sentence-transformers

# Vector database
!pip install faiss-cpu==1.7.4

# Open-source LLM
!pip install transformers torch accelerate

# Hybrid retriever
!pip install rank-bm25

In [2]:
# Import libraries

import requests
import pandas as pd
import numpy as np
import pickle
import faiss

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import pipeline
from pprint import pprint

C:\Users\Chidiebere\anaconda3\envs\nsia_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Ingestion

>  Public Nigeria state dataset

In [3]:
API_URL = "https://nga-states-lga.onrender.com/fetch"

response = requests.get(API_URL)

# Convert JSON response into Python object
state_data = response.json()

print(type(state_data))
print(len(state_data))

<class 'list'>
37


In [4]:
# Inspect API data - understand the schema

pprint(state_data[0])

'Abia'


### Convert API data to Dataframe
> For easier preprocessing

In [12]:
state_df = pd.DataFrame(state_data)

state_df.head()

,0
0,Abia
1,Adamawa
2,AkwaIbom
3,Anambra
4,Bauchi


In [7]:
# Inspect columns

print(state_df.columns)

RangeIndex(start=0, stop=1, step=1)


### Create RAG Documents

RAG systems work better with natural language documents, although the source data is structured JSON, we will transform it into descriptive text. This process is called: Structured-to-unstructured transformation.

In [10]:
rag_documents = []

for _, row in state_df.iterrows():
    
    document = f"""
    State Name: {row.get('state')}
    
    Capital: {row.get('capital')}
    
    Region: {row.get('region')}
    
    Governor: {row.get('governor')}
    
    Slogan: {row.get('slogan')}
    
    Latitude: {row.get('latitude')}
    
    Longitude: {row.get('longitude')}
    
    Description: {row.get('description')}
    
    """
    
    rag_documents.append(document)
    
print(rag_documents[0])


    State Name: None
    
    Capital: None
    
    Region: None
    
    Governor: None
    
    Slogan: None
    
    Latitude: None
    
    Longitude: None
    
    Description: None
    
    


In [16]:
import requests

states_url = "https://nga-states-lga.onrender.com/fetch"

states = requests.get(states_url).json()

documents = []

failed_states = []

for state in states:

    # Fix problematic spellings
    corrected_state = state.replace(
        "AkwaIbom",
        "Akwa Ibom"
    )

    lga_url = f"https://nga-states-lga.onrender.com/?state={corrected_state}"

    response = requests.get(lga_url)

    try:

        lga_data = response.json()

        # Ensure lga is a list
        if isinstance(lga_data, list):

            document = f"""
            State: {corrected_state}

            Local Government Areas:
            {', '.join(lgas)}
            """

            documents.append(document)

        else:

            failed_states.append(corrected_state)

    except Exception as e:

        failed_states.append(corrected_state)

print(f"Documents created: {len(documents)}")

print(f"Failed states: {failed_states}")

print(documents[0])

Documents created: 0
Failed states: ['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Rivers', 'Enugu', 'FCT', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']


IndexError: list index out of range

In [17]:
# The API does not accept some state names directly from /fetch. Results to manual

states = [
    "Abia",
    "Adamawa",
    "Akwa Ibom",
    "Anambra",
    "Bauchi",
    "Bayelsa",
    "Benue",
    "Borno",
    "Cross River",
    "Delta",
    "Ebonyi",
    "Edo",
    "Ekiti",
    "Enugu",
    "FCT",
    "Gombe",
    "Imo",
    "Jigawa",
    "Kaduna",
    "Kano",
    "Katsina",
    "Kebbi",
    "Kogi",
    "Kwara",
    "Lagos",
    "Nasarawa",
    "Niger",
    "Ogun",
    "Ondo",
    "Osun",
    "Oyo",
    "Plateau",
    "Rivers",
    "Sokoto",
    "Taraba",
    "Yobe",
    "Zamfara"
]

documents = []

failed_states = []

for state in states:

    url = (
        f"https://nga-states-lga.onrender.com/?state={state}"
    )

    response = requests.get(url)

    try:

        lgas = response.json()

        if isinstance(lgas, list):

            document = f"""
            State: {state}

            Local Government Areas:
            {', '.join(lgas)}
            """

            documents.append(document)

        else:

            failed_states.append(state)

    except Exception:

        failed_states.append(state)

print(f"Documents created: {len(documents)}")

print(f"Failed states: {failed_states}")

print(documents[0])

Documents created: 36
Failed states: ['Akwa Ibom']

            State: Abia

            Local Government Areas:
            Aba North, Aba South, Arochukwu, Bende, Ikwuano, Isiala Ngwa North, Isiala Ngwa South, Isuikwuato, Obi Ngwa, Ohafia, Osisioma, Ugwunagbo, Ukwa East, Ukwa West, Umuahia North, Umuahia South, Umu Nneochi
            


In [23]:
# API Verification Framework

import requests
from pprint import pprint

def inspect_api(name, url):
    
    try:
        response = requests.get(url, timeout=20)
        print(f" status code: {response.status_code}")
        
        data = response.json()
        
        print(f"data type: {type(data)}")
        
        
        # List Response
        if isinstance(data, list):
            
            print(f"List length: {len(data)}")
            
            pprint(data[0])
            
            
        # Dictionary Response
        elif isinstance(data, dict):
            pprint(list(data.keys())[:20])
            
            pprint(data)
            
        else:
            print(data)
            
    except Exception as e:
        
        print(f"FAILED: {e}")

### STATES + LGAs API

In [24]:
inspect_api(
    "Nigeria States API",
    "https://nga-states-lga.onrender.com/?state=Lagos"
)

 status code: 200
data type: <class 'list'>
List length: 20
'Agege'


### UNIVERSITIES API

In [25]:
inspect_api(
    "Nigeria Universities API",
    "http://universities.hipolabs.com/search?country=Nigeria"
)

 status code: 200
data type: <class 'list'>
List length: 115
{'alpha_two_code': 'NG',
 'country': 'Nigeria',
 'domains': ['aaua.edu.ng'],
 'name': 'Adekunle Ajasin University',
 'state-province': None,
 'web_pages': ['http://www.aaua.edu.ng/']}


### EXCHANGE RATE API

In [26]:
inspect_api(
    "Exchange Rate API",
    "https://api.exchangerate.host/latest?base=USD&symbols=NGN"
)

 status code: 200
data type: <class 'dict'>
['success', 'error']
{'error': {'code': 101,
           'info': 'You have not supplied an API Access Key. [Required format: '
                   'access_key=YOUR_ACCESS_KEY]',
           'type': 'missing_access_key'},
 'success': False}
